# Tahap 3 — Eksperimen Clustering & Analisis Silhouette

Tahap ini adalah inti dari analisis: menjalankan **K-Means clustering** pada semua representasi data (8 variasi dari pipeline 204→203→74→37), mengevaluasi setiap kombinasi dengan **Silhouette Coefficient**, lalu memilih konfigurasi terbaik.

---

## Konsep Utama

### K-Means Clustering

K-Means mengelompokkan $n$ titik data ke dalam $k$ cluster dengan meminimalkan **Within-Cluster Sum of Squares (WCSS)**:

$$J = \sum_{j=1}^{k} \sum_{x_i \in C_j} \|x_i - \mu_j\|^2$$

di mana $\mu_j$ adalah centroid cluster $C_j$.

**Algoritma:**
1. Inisialisasi $k$ centroid (metode `k-means++` untuk inisialisasi lebih cerdas)
2. Tetapkan tiap titik ke centroid terdekat
3. Perbarui centroid sebagai rata-rata cluster
4. Ulangi 2–3 hingga konvergen

### Silhouette Coefficient

Untuk setiap titik data $i$:

$$s(i) = \frac{b(i) - a(i)}{\max(a(i),\, b(i))} \in [-1, +1]$$

- $a(i)$ = jarak rata-rata ke semua titik **dalam clusternya** sendiri *(intra-cluster cohesion)*
- $b(i)$ = jarak rata-rata ke semua titik **di cluster terdekat** *(inter-cluster separation)*

| Nilai | Interpretasi |
|:-----:|:-------------|
| dekat **+1** | Titik berada jauh dari cluster lain → cluster sangat baik |
| dekat **0**  | Titik berada di perbatasan dua cluster |
| dekat **−1** | Titik mungkin salah diklasifikasi |

### Elbow Method

Grafik **Inertia vs k** — mencari titik 'siku' di mana penambahan cluster tidak lagi menurunkan inertia secara signifikan. Digunakan bersama Silhouette untuk konfirmasi.

---

## Desain Eksperimen

| Parameter | Nilai |
|:----------|:------|
| Dataset   | 8 (Linear & Poly × 204, 203, 74, 37 dimensi) |
| Nilai *k* | 2, 3, 4, 5, 6, 7, 8, 9, 10 |
| Total run | **72 kombinasi** |
| Algoritma | K-Means (`n_init=20`, `random_state=42`) |
| Metrik    | Silhouette Score + Inertia |

---

## Hasil Clustering untuk Setiap Dimensi

Pipeline reduksi dimensi menghasilkan empat tahap: **204 → 203 → 74 → 37**. Pada tahap ini **setiap dimensi diberi hasil clustering sendiri**: untuk tiap dimensi (dan tiap metode interpolasi, Linear & Poly) dicari **k terbaik** berdasarkan Silhouette Coefficient, lalu hasilnya dirangkum dalam satu tabel dan dipetakan pada Tahap 4.


## 3.1 Import & Load Data Tahap 2

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pickle, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

OUTPUT_DIR = '../output'
plt.rcParams['figure.figsize'] = (13, 5)
sns.set_theme(style='whitegrid', font_scale=1.05)

# ── Load data dari Tahap 2 ───────────────────────────────────
with open(f'{OUTPUT_DIR}/tahap2_pca.pkl', 'rb') as f:
    d = pickle.load(f)

representasi = d['representasi']
meta_linear  = d['meta_linear']
meta_poly    = d['meta_poly']
X_linear     = d['X_linear']
X_poly       = d['X_poly']

print('Data berhasil dimuat.')
print(f'Total representasi : {len(representasi)}')
print('\nRepresentasi yang tersedia:')
for nama, X in representasi.items():
    print(f'  {nama:<15} → shape: {X.shape}')

## 3.2 Jalankan 72 Eksperimen Clustering

Kita menguji semua kombinasi: **8 representasi × 9 nilai k** = 72 eksperimen.

In [ ]:
# ── Parameter eksperimen ─────────────────────────────────────
K_RANGE      = range(2, 11)   # k dari 2 sampai 10
N_INIT       = 20             # Lebih banyak inisialisasi → lebih stabil
RANDOM_STATE = 42

hasil = []        # Kumpulkan semua hasil
labels_map = {}   # Simpan label untuk setiap (dataset, k)

total = len(representasi) * len(K_RANGE)
run   = 0

print(f'Menjalankan {total} eksperimen...')
print('=' * 65)

for nama_data, X in representasi.items():
    best_sil, best_k = -1, 2
    for k in K_RANGE:
        run += 1
        km     = KMeans(n_clusters=k, n_init=N_INIT,
                        init='k-means++', random_state=RANDOM_STATE)
        labels = km.fit_predict(X)
        # Urutkan nomor cluster dari yang terbesar (C0 = cluster terbesar) agar
        # nomor/warna konsisten antar dimensi. Silhouette & inertia tidak berubah.
        urut  = np.argsort(-np.bincount(labels, minlength=k), kind='stable')
        remap = np.empty(k, dtype=int); remap[urut] = np.arange(k)
        labels = remap[labels]
        sil    = silhouette_score(X, labels)
        iner   = km.inertia_

        labels_map[(nama_data, k)] = labels
        hasil.append({
            'dataset'   : nama_data,
            'k'         : k,
            'silhouette': round(sil, 6),
            'inertia'   : round(iner, 4),
        })
        if sil > best_sil:
            best_sil, best_k = sil, k

    print(f'  {nama_data:<15}  k_terbaik = {best_k}   silhouette = {best_sil:.4f}')

df_hasil = pd.DataFrame(hasil)
print(f'\nSelesai! Total {run} eksperimen dijalankan.')

## 3.3 Visualisasi Silhouette Score vs k (semua dataset)

Tiap subplot menampilkan kurva Silhouette untuk satu representasi data. Titik **bintang emas** menandai nilai k terbaik.

In [ ]:
# ── Warna per dataset ─────────────────────────────────────────
PALETTE = {
    'Linear-204': '#0D47A1', 'Linear-203': '#1565C0',
    'Linear-74' : '#1976D2', 'Linear-37' : '#90CAF9',
    'Poly-204'  : '#B71C1C', 'Poly-203'  : '#C62828',
    'Poly-74'   : '#E53935', 'Poly-37'   : '#FFCDD2',
}

fig, axes = plt.subplots(2, 4, figsize=(22, 11))
axes = axes.flatten()

for idx, nama_data in enumerate(representasi):
    sub   = df_hasil[df_hasil['dataset'] == nama_data]
    color = PALETTE.get(nama_data, '#555555')
    ax    = axes[idx]

    ax.plot(sub['k'], sub['silhouette'], 'o-', color=color,
            linewidth=2.5, markersize=9, zorder=3)
    ax.fill_between(sub['k'], sub['silhouette'], alpha=0.12, color=color)

    # Tandai titik terbaik
    best = sub.loc[sub['silhouette'].idxmax()]
    ax.scatter(best['k'], best['silhouette'], s=280,
               color='gold', edgecolors=color, linewidth=2.5,
               zorder=5, marker='*', label=f'k={int(best["k"])} (terbaik)')
    ax.annotate(f"  k={int(best['k'])}\n  sil={best['silhouette']:.3f}",
                xy=(best['k'], best['silhouette']), fontsize=9,
                fontweight='bold', color=color)

    ax.set_title(f'{nama_data}', fontweight='bold', fontsize=11)
    ax.set_xlabel('Jumlah Cluster (k)')
    ax.set_ylabel('Silhouette Score')
    ax.set_xticks(list(K_RANGE))
    ax.legend(fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(True, alpha=0.3)

plt.suptitle('Silhouette Score vs Jumlah Cluster (k)\nuntuk 8 Representasi Data (Pipeline 204→203→74→37)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/07_silhouette_semua.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gambar disimpan: output/07_silhouette_semua.png')

## 3.4 Tabel Perbandingan & Pemilihan Konfigurasi Terbaik

In [ ]:
# ── Tabel ringkasan best k per dataset ───────────────────────
ringkasan = (
    df_hasil.loc[df_hasil.groupby('dataset', sort=False)['silhouette'].idxmax()]
    [['dataset', 'k', 'silhouette', 'inertia']]
    .sort_values('silhouette', ascending=False)
    .reset_index(drop=True)
)
ringkasan.index += 1
ringkasan.columns = ['Dataset', 'k Terbaik', 'Silhouette', 'Inertia']

print('PERBANDINGAN KONFIGURASI TERBAIK TIAP DATASET:')
print('=' * 65)
print(ringkasan.to_string())
ringkasan.to_csv(f'{OUTPUT_DIR}/ringkasan_eksperimen.csv')

# ── Pilih GLOBAL TERBAIK ──────────────────────────────────────
best_row = df_hasil.loc[df_hasil['silhouette'].idxmax()]
BEST_DATASET = best_row['dataset']
BEST_K       = int(best_row['k'])
BEST_SIL     = best_row['silhouette']
BEST_LABELS  = labels_map[(BEST_DATASET, BEST_K)]

print(f'\n>> KONFIGURASI TERBAIK GLOBAL:')
print(f'   Dataset    : {BEST_DATASET}')
print(f'   k (cluster): {BEST_K}')
print(f'   Silhouette : {BEST_SIL:.4f}')

## 3.4b Cluster Terbaik untuk Setiap Dimensi (204 → 203 → 74 → 37)

Untuk **setiap dimensi** dipilih k dengan Silhouette tertinggi (k = 2 sampai 10), terpisah untuk interpolasi **Linear** dan **Polynomial**. Tabel ini adalah jawaban untuk "setiap dimensi diberi hasil clustering".

In [ ]:
from sklearn.metrics import adjusted_rand_score

DIMENSI = [204, 203, 74, 37]     # urutan pipeline reduksi
METODE  = ['Linear', 'Poly']

# ── Cluster terbaik untuk SETIAP dimensi ─────────────────────
BEST_PER_DIM = {}                # kunci: (metode, dimensi)
for metode in METODE:
    for dim in DIMENSI:
        nama = f'{metode}-{dim}'
        if nama not in representasi:
            continue
        sub = df_hasil[df_hasil['dataset'] == nama]
        row = sub.loc[sub['silhouette'].idxmax()]
        k   = int(row['k'])
        lab = labels_map[(nama, k)]
        BEST_PER_DIM[(metode, dim)] = {
            'dataset'   : nama,
            'metode'    : metode,
            'dimensi'   : dim,
            'k'         : k,
            'silhouette': float(row['silhouette']),
            'inertia'   : float(row['inertia']),
            'labels'    : lab,
            'ukuran'    : np.bincount(lab, minlength=k).tolist(),
        }

# ── Tabel ringkasan, beri catatan jika hasil sama dengan dimensi sebelumnya ──
baris = []
for metode in METODE:
    prev = None
    for dim in DIMENSI:
        info = BEST_PER_DIM.get((metode, dim))
        if info is None:
            continue
        catatan = ''
        if (prev is not None and prev['k'] == info['k']
                and adjusted_rand_score(prev['labels'], info['labels']) > 0.9999):
            catatan = f"partisi sama dengan {prev['dataset']}"
        baris.append({
            'Metode'        : metode,
            'Dimensi'       : dim,
            'k terbaik'     : info['k'],
            'Silhouette'    : round(info['silhouette'], 4),
            'Inertia'       : round(info['inertia'], 1),
            'Ukuran cluster': ' / '.join(str(u) for u in info['ukuran']),
            'Catatan'       : catatan,
        })
        prev = info
df_dim = pd.DataFrame(baris)

print('CLUSTER TERBAIK UNTUK SETIAP DIMENSI')
print('=' * 95)
print(df_dim.to_string(index=False))
df_dim.to_csv(f'{OUTPUT_DIR}/ringkasan_per_dimensi.csv', index=False)

# ── Juara tiap dimensi: Linear vs Poly ──────────────────────
print('\nPEMENANG TIAP DIMENSI (Linear vs Poly):')
for dim in DIMENSI:
    cands = [BEST_PER_DIM[(m, dim)] for m in METODE if (m, dim) in BEST_PER_DIM]
    win = max(cands, key=lambda x: x['silhouette'])
    print(f"  {dim:>3} dimensi -> {win['dataset']:<11} k={win['k']}  silhouette={win['silhouette']:.4f}")

In [ ]:
# ── Grafik: silhouette terbaik di tiap tahap reduksi ──────────
fig, ax = plt.subplots(figsize=(11, 6))
xs = np.arange(len(DIMENSI))
for metode, color in [('Linear', '#1565C0'), ('Poly', '#B71C1C')]:
    pts = [(i, BEST_PER_DIM[(metode, d)]) for i, d in enumerate(DIMENSI) if (metode, d) in BEST_PER_DIM]
    ax.plot([i for i, _ in pts], [p['silhouette'] for _, p in pts], 'o-',
            color=color, linewidth=2.5, markersize=10, label=metode)
    for i, p in pts:
        ax.annotate(f"k={p['k']}\n{p['silhouette']:.3f}", xy=(i, p['silhouette']),
                    xytext=(0, 12 if metode == 'Poly' else -30), textcoords='offset points',
                    ha='center', fontsize=9, fontweight='bold', color=color)

ax.set_xticks(xs)
ax.set_xticklabels([f'{d}\ndimensi' for d in DIMENSI])
ax.set_xlabel('Tahap reduksi dimensi  (204 → 203 → 74 → 37)', fontsize=11)
ax.set_ylabel('Silhouette terbaik (k = 2..10)', fontsize=11)
ax.set_title('Cluster Terbaik di Setiap Dimensi', fontweight='bold', fontsize=13)
ax.margins(y=0.25)
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/07c_silhouette_per_dimensi.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gambar disimpan: output/07c_silhouette_per_dimensi.png')

**Cara membaca kolom Catatan:** jika sebuah dimensi menghasilkan *partisi yang sama persis* dengan dimensi sebelumnya, itu bukan kesalahan. Dua penyebab yang wajar:

- **204 → 203:** satu fitur yang dibuang adalah fitur konstan (nilainya sama di semua wilayah), sehingga tidak memengaruhi jarak antar wilayah.
- **74 → 37:** dengan hanya 37 wilayah, PCA 37 komponen menangkap **100% variansi** (hanya memutar sumbu tanpa membuang informasi), sehingga jarak antar wilayah tetap sama.

Kolom *Ukuran cluster* menunjukkan jumlah wilayah per cluster. Perhatikan jika ada cluster yang sangat kecil: nilai silhouette yang tinggi pada k=2 kadang berasal dari beberapa wilayah yang menyimpang jauh dari yang lain (outlier).

## 3.5 Heatmap Perbandingan Silhouette — Semua Dimensi

Heatmap ini merangkum seluruh 72 eksperimen dalam satu visualisasi. Baris = dataset, Kolom = k.

In [ ]:
# ── Heatmap Silhouette Score ──────────────────────────────────
pivot = df_hasil.pivot(index='dataset', columns='k', values='silhouette')

# Urutkan baris sesuai urutan pipeline
urutan = ['Linear-204', 'Linear-203', 'Linear-74', 'Linear-37',
          'Poly-204',   'Poly-203',   'Poly-74',   'Poly-37']
pivot = pivot.reindex([u for u in urutan if u in pivot.index])

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(
    pivot, annot=True, fmt='.3f',
    cmap='RdYlGn', center=0.3,
    linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Silhouette Score', 'shrink': 0.7}
)

# Tandai sel terbaik
best_row_idx = list(pivot.index).index(BEST_DATASET)
best_col_idx = list(pivot.columns).index(BEST_K)
ax.add_patch(plt.Rectangle((best_col_idx, best_row_idx), 1, 1,
                             fill=False, edgecolor='gold', lw=3))

ax.set_title(f'Heatmap Silhouette Score — 8 Dataset × 9 Nilai k\n'
             f'[Kotak emas = konfigurasi terbaik: {BEST_DATASET}, k={BEST_K}, sil={BEST_SIL:.3f}]',
             fontweight='bold', fontsize=12)
ax.set_xlabel('Jumlah Cluster (k)', fontsize=11)
ax.set_ylabel('Dataset', fontsize=11)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/07b_heatmap_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gambar disimpan: output/07b_heatmap_silhouette.png')

## 3.6 Silhouette Plot — Konfigurasi Terbaik

Silhouette plot menampilkan nilai silhouette **per sampel** dalam setiap cluster, diurutkan dari terbesar ke terkecil. Visualisasi ini mengungkap:
- Seberapa konsisten tiap cluster (lebar/homogenitas tiap blok)
- Apakah ada sampel yang "salah masuk" cluster (nilai negatif)

In [ ]:
# ── Silhouette Plot ───────────────────────────────────────────
X_best     = representasi[BEST_DATASET]
sil_vals   = silhouette_samples(X_best, BEST_LABELS)
colors_tab = plt.cm.tab10(np.linspace(0, 1, BEST_K))

fig, ax = plt.subplots(figsize=(10, 7))
y_lower = 10

for i in range(BEST_K):
    vals_i  = np.sort(sil_vals[BEST_LABELS == i])
    size_i  = len(vals_i)
    y_upper = y_lower + size_i

    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, vals_i,
                     facecolor=colors_tab[i], alpha=0.85)
    ax.text(-0.07, y_lower + size_i / 2,
            f'C{i}\n({size_i})', fontsize=9.5, va='center', fontweight='bold')
    y_lower = y_upper + 8

# Garis rata-rata
ax.axvline(x=BEST_SIL, color='crimson', linestyle='--', linewidth=2,
           label=f'Rata-rata = {BEST_SIL:.3f}')
ax.axvline(x=0, color='black', linewidth=0.8)

ax.set_xlabel('Silhouette Coefficient', fontsize=12)
ax.set_ylabel('Sampel (per cluster)', fontsize=12)
ax.set_title(f'Silhouette Plot — {BEST_DATASET}, k={BEST_K}\n'
             f'Silhouette Score = {BEST_SIL:.4f}',
             fontweight='bold', fontsize=13)
ax.set_yticks([])
ax.set_xlim(-0.2, 1.0)
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/08_silhouette_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gambar disimpan: output/08_silhouette_plot.png')

## 3.6b Silhouette Plot — Semua Dimensi

Silhouette plot per sampel untuk **k terbaik di setiap dimensi** (baris atas Linear, baris bawah Polynomial).

In [ ]:
tab = plt.cm.tab10(np.linspace(0, 1, 10))
fig, axes = plt.subplots(2, 4, figsize=(22, 10))

for r, metode in enumerate(METODE):
    for ci, dim in enumerate(DIMENSI):
        ax   = axes[r, ci]
        info = BEST_PER_DIM.get((metode, dim))
        if info is None:
            ax.axis('off')
            continue
        X, lab, k = representasi[info['dataset']], info['labels'], info['k']
        sv = silhouette_samples(X, lab)
        y_lower = 5
        for i in range(k):
            v = np.sort(sv[lab == i])
            y_upper = y_lower + len(v)
            ax.fill_betweenx(np.arange(y_lower, y_upper), 0, v, facecolor=tab[i % 10], alpha=0.85)
            ax.text(-0.04, y_lower + len(v) / 2, f'C{i} ({len(v)})', fontsize=8, va='center', ha='right')
            y_lower = y_upper + 3
        ax.axvline(info['silhouette'], color='crimson', linestyle='--', linewidth=2)
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_xlim(-0.35, 1.0)
        ax.set_yticks([])
        ax.set_title(f"{info['dataset']}  |  k={k}  |  sil={info['silhouette']:.3f}", fontweight='bold', fontsize=10.5)
        ax.set_xlabel('Silhouette Coefficient')
        ax.spines[['top', 'right', 'left']].set_visible(False)

plt.suptitle('Silhouette Plot Cluster Terbaik di Setiap Dimensi (204 → 203 → 74 → 37)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/08b_silhouette_plot_per_dimensi.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gambar disimpan: output/08b_silhouette_plot_per_dimensi.png')

## 3.7 Elbow Method — Konfirmasi k Terbaik

In [ ]:
# ── Elbow Method ─────────────────────────────────────────────
sub_best = df_hasil[df_hasil['dataset'] == BEST_DATASET]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Elbow (Inertia)
ax = axes[0]
ax.plot(sub_best['k'], sub_best['inertia'], 'o-',
        color='#1565C0', linewidth=2.5, markersize=8)
ax.fill_between(sub_best['k'], sub_best['inertia'], alpha=0.1, color='#1565C0')
ax.axvline(BEST_K, color='crimson', linestyle='--', linewidth=2,
           label=f'k={BEST_K} (terpilih)')
ax.set_title(f'Elbow Method (Inertia)\n{BEST_DATASET}', fontweight='bold', fontsize=12)
ax.set_xlabel('Jumlah Cluster (k)')
ax.set_ylabel('Inertia (WCSS)')
ax.set_xticks(list(K_RANGE))
ax.legend()
ax.spines[['top','right']].set_visible(False)
ax.grid(True, alpha=0.3)

# Silhouette
ax = axes[1]
ax.plot(sub_best['k'], sub_best['silhouette'], 'o-',
        color='#B71C1C', linewidth=2.5, markersize=8)
ax.fill_between(sub_best['k'], sub_best['silhouette'], alpha=0.1, color='#B71C1C')
ax.scatter(BEST_K, BEST_SIL, s=200, color='gold',
           edgecolors='#B71C1C', linewidth=2, zorder=5,
           label=f'k={BEST_K}: sil={BEST_SIL:.3f}')
ax.set_title(f'Silhouette Score\n{BEST_DATASET}', fontweight='bold', fontsize=12)
ax.set_xlabel('Jumlah Cluster (k)')
ax.set_ylabel('Silhouette Score')
ax.set_xticks(list(K_RANGE))
ax.legend()
ax.spines[['top','right']].set_visible(False)
ax.grid(True, alpha=0.3)

plt.suptitle(f'Elbow Method & Silhouette — Konfirmasi k={BEST_K}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/09_elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.8 Distribusi Cluster & Scatter PCA 2D

In [ ]:
# ── Scatter PCA 2D + Pie Chart ───────────────────────────────
from sklearn.decomposition import PCA as PCA_viz

CLUSTER_COLORS = [
    '#E53935','#1E88E5','#43A047','#FB8C00','#8E24AA',
    '#00ACC1','#F06292','#FDD835','#6D4C41','#78909C'
]

pca2d  = PCA_viz(n_components=2, random_state=42)
X_viz  = pca2d.fit_transform(X_best)

meta_best = meta_linear if 'Linear' in BEST_DATASET else meta_poly

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter
ax = axes[0]
for c in range(BEST_K):
    mask = BEST_LABELS == c
    ax.scatter(X_viz[mask, 0], X_viz[mask, 1], s=140,
               color=CLUSTER_COLORS[c], edgecolors='white',
               linewidth=1.5, label=f'Cluster {c}', alpha=0.9, zorder=3)
    for idx in np.where(mask)[0]:
        label_txt = meta_best.iloc[idx]['daerah'].split(',')[0][:12]
        ax.annotate(label_txt,
                    xy=(X_viz[idx,0], X_viz[idx,1]),
                    xytext=(0, 8), textcoords='offset points',
                    fontsize=6, ha='center', color='#333')

ax.set_title(f'Proyeksi PCA 2D — Hasil Clustering\n{BEST_DATASET}, k={BEST_K}',
             fontweight='bold', fontsize=12)
ax.set_xlabel(f'PC1 ({pca2d.explained_variance_ratio_[0]:.1%} variansi)')
ax.set_ylabel(f'PC2 ({pca2d.explained_variance_ratio_[1]:.1%} variansi)')
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)
ax.grid(True, alpha=0.3)

# Pie
ax2 = axes[1]
uniq, cnts = np.unique(BEST_LABELS, return_counts=True)
ax2.pie(cnts,
        labels=[f'Cluster {u}\n({c} wilayah)' for u,c in zip(uniq,cnts)],
        colors=[CLUSTER_COLORS[u] for u in uniq],
        autopct='%1.1f%%', startangle=90,
        wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2),
        textprops={'fontsize': 10})
ax2.set_title(f'Distribusi 37 Wilayah per Cluster\n(k={BEST_K})',
              fontweight='bold', fontsize=12)

plt.suptitle('Visualisasi Hasil Clustering Terbaik', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/10_scatter_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.9 Profil Cluster — Heatmap Rata-rata Fitur

In [ ]:
# ── Profil rata-rata fitur kunci per cluster ──────────────────
X_raw = X_linear if 'Linear' in BEST_DATASET else X_poly
key_cols, key_labels = [], []
for pol, pol_lbl in [('no2','NO₂'), ('so2','SO₂'), ('co','CO')]:
    for feat, feat_lbl in [('calc_mean','Mean'), ('calc_std','Std'),
                            ('entropy','Entropy'), ('skewness','Skewness')]:
        col = f'{pol}_{feat}'
        if col in X_raw.columns:
            key_cols.append(col)
            key_labels.append(f'{pol_lbl} {feat_lbl}')

df_profil = X_raw[key_cols].copy()
df_profil['cluster'] = BEST_LABELS
profil = df_profil.groupby('cluster')[key_cols].mean()

# Normalisasi 0-1 per fitur (agar warna dapat dibandingkan)
profil_norm = (profil - profil.min()) / (profil.max() - profil.min() + 1e-10)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    profil_norm.T,
    annot=profil.T.round(3),   # Tampilkan nilai asli, bukan normalisasi
    fmt='.3f', cmap='YlOrRd',
    linewidths=0.5, ax=ax,
    xticklabels=[f'Cluster {c}' for c in profil.index],
    yticklabels=key_labels,
    cbar_kws={'label': 'Nilai Ternormalisasi (0→1)', 'shrink': 0.7}
)
ax.set_title(f'Profil Polutan per Cluster — {BEST_DATASET}, k={BEST_K}\n'
             f'(Nilai di sel = rata-rata asli, warna = nilai normalisasi)',
             fontweight='bold', fontsize=12)
ax.set_xlabel('Cluster')
ax.set_ylabel('Fitur Polutan')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/11_profil_cluster.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gambar disimpan: output/11_profil_cluster.png')

In [ ]:
# ── Simpan hasil untuk Tahap 4 ────────────────────────────────
save_data = {
    'BEST_DATASET' : BEST_DATASET,
    'BEST_K'       : BEST_K,
    'BEST_SIL'     : BEST_SIL,
    'BEST_LABELS'  : BEST_LABELS,
    'CLUSTER_COLORS': CLUSTER_COLORS,
    'meta_linear'  : meta_linear,
    'meta_poly'    : meta_poly,
    'X_linear'     : X_linear,
    'X_poly'       : X_poly,
    'df_hasil'     : df_hasil,
    'ringkasan'    : ringkasan,
    # hasil cluster terbaik untuk SETIAP dimensi (204 -> 203 -> 74 -> 37)
    'BEST_PER_DIM' : BEST_PER_DIM,
    'DIMENSI'      : DIMENSI,
    'ringkasan_dim': df_dim,
}
save_path = f'{OUTPUT_DIR}/tahap3_clustering.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(save_data, f)

print(f'Hasil clustering disimpan: {os.path.abspath(save_path)}')
print(f'Ukuran file               : {os.path.getsize(save_path)/1024:.1f} KB')
print('\nDistribusi wilayah per cluster:')
meta_best2 = meta_linear if 'Linear' in BEST_DATASET else meta_poly
for c in range(BEST_K):
    wil = meta_best2[BEST_LABELS == c]['daerah'].tolist()
    print(f'  Cluster {c} ({len(wil)} wilayah): {", ".join(wil)}')
print('\n→ Lanjut ke Tahap 4: Visualisasi Peta Segmentasi')